In [1]:
import pandas as pd
#from webdriver_manager.chrome import ChromeDriverManager
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from subprocess import CREATE_NO_WINDOW
from webdriver_manager.core.os_manager import OperationSystemManager,ChromeType

In [2]:
url = 'https://stock.naver.com/domestic/stock/069500/info/summary'

In [3]:
br_ver = OperationSystemManager().get_browser_version_from_os(ChromeType.GOOGLE)
version_main=int(br_ver.split('.')[0])
'Dirver Setting'

option = Options()
option.add_argument('--disable-gpu')
option.add_argument('--window-size=1920x1080')
option.add_argument('--start-maximized')
option.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36')
service = Service()
service.creation_flags = CREATE_NO_WINDOW

# driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver = uc.Chrome(service=service, options=option, version_main=version_main)
driver.implicitly_wait(3) # 화명 렌더링을 3초간 기다림
str1 = driver.capabilities['browserVersion']
str2 = driver.capabilities['chrome']['chromedriverVersion'].split(' ')[0]

print(f'chrome browser version : {str1[0:2]}, chrome dirver version : {str2[0:2]}')

driver.get(url)
# divs = driver.find_elements(By.CLASS_NAME, "StockInfo_listing-info__qzcRk")

li_elements = driver.find_elements(By.XPATH, '//ul[@class="StockInfo_listing-info__qzcRk"]/li') # //ul[@class="StockInfo_listing-info__qzcRk"]/li

results = []

for li in li_elements:
    # print(li)
    spans = li.find_elements(By.TAG_NAME, "span")
    data = [span.text for span in spans if span.text.strip() != ""]
    if data:
        item = {
            "지표": data[0],
            "값": data[1] if len(data) > 1 else 'N/A'
        }
        results.append(item) # 리스트에 추가
        # print(f"지표: {item['지표']}, 값: {item['값']}") # 기존 출력 유지


# 1. 특정 캡션(포트폴리오 구성)을 포함한 테이블을 먼저 찾습니다.
# "069500 포트폴리오 구성" 텍스트가 포함된 caption의 부모 table을 타겟팅합니다.
# target_table = driver.find_element(By.XPATH, "//table[contains(caption, '포트폴리오 구성')]")
# target_table_xpath = "//table[contains(caption, '포트폴리오 구성')]/following::table[@class='InnerTable_table___xmXR'][1]"
# target_table = driver.find_element(By.XPATH, target_table_xpath)

all_tables = driver.find_elements(By.CSS_SELECTOR, "table.InnerTable_table___xmXR")
table_results = []
seen_names = set()  
if len(all_tables) >= 2:
    target_table = all_tables[0] 
    rows = target_table.find_elements(By.CSS_SELECTOR, "tbody.InnerTable_tbody__zuUyv tr")
    
    row_num = 0
    for row in rows:
        if row_num <= 10:
            cells = row.find_elements(By.TAG_NAME, "td")
            print([cell.text for cell in cells])
            # raw_data = [cell.text for cell in cells]
            row_data = [cell.text.replace('\n', ' ').strip() for cell in cells]
            stock_name = row_data[0]
            if stock_name not in seen_names:

                item = {
                    "종목명": row_data[0],
                    "주식수": row_data[1],
                    "비중중": row_data[2],
                    "시세": row_data[3] if len(row_data) > 3 else 'N/A',
                    "전일대비": row_data[4] if len(row_data) > 4 else 'N/A'
                }
                table_results.append(item)
                seen_names.add(stock_name)

            row_num += 1
            print(row_num)

driver.quit()

chrome browser version : 14, chrome dirver version : 14
['삼성전자', '7,022\n주', '32.90\n%', '290,500', '상승\n5,000\n(+1.75%)']
1
['SK하이닉스', '834\n주', '25.73\n%', '1,957,000', '상승\n77,000\n(+4.10%)']
2
['SK스퀘어', '139\n주', '2.70\n%', '1,223,000', '상승\n36,000\n(+3.03%)']
3
['현대차', '205\n주', '2.17\n%', '671,500', '상승\n25,500\n(+3.95%)']
4
['두산에너빌리티', '653\n주', '1.37\n%', '132,900', '상승\n4,900\n(+3.83%)']
5
['KB금융', '482\n주', '1.25\n%', '158,800', '0\n(0.00%)']
6
['삼성전기', '84\n주', '1.24\n%', '935,000', '상승\n35,000\n(+3.89%)']
7
['삼성물산', '145\n주', '1.07\n%', '461,000', '상승\n9,000\n(+1.99%)']
8
['한화에어로스페이스', '49\n주', '1.05\n%', '1,305,000', '하락\n10,000\n(-0.76%)']
9
['기아', '368\n주', '1.05\n%', '174,400', '하락\n300\n(-0.17%)']
10


In [5]:
print(results)

[{'지표': '기초지수\n(추적지수)', '값': '코스피 200'}, {'지표': '상장일', '값': '2002. 10. 14.'}, {'지표': '운용사', '값': '삼성자산운용(주)'}, {'지표': '시가총액', '값': '26조 4,133억'}, {'지표': '운용자산', '값': '25조 8,698억'}, {'지표': '레버리지', '값': '1배'}, {'지표': 'NAV', '값': '121,854'}, {'지표': '괴리율', '값': '-0.04%'}, {'지표': '총보수', '값': '0.15%'}, {'지표': '추적오차율', '값': '0.40%'}, {'지표': '증권거래', '값': '-'}, {'지표': '매매차익', '값': '비과세'}, {'지표': '배당소득세', '값': '-'}]


In [8]:
results_agg = dict()
results_agg['001'] = results
results_agg['002'] = results

In [20]:
results_agg['001']

[{'지표': '기초지수\n(추적지수)', '값': '코스피 200'},
 {'지표': '상장일', '값': '2002. 10. 14.'},
 {'지표': '운용사', '값': '삼성자산운용(주)'},
 {'지표': '시가총액', '값': '26조 4,133억'},
 {'지표': '운용자산', '값': '25조 8,698억'},
 {'지표': '레버리지', '값': '1배'},
 {'지표': 'NAV', '값': '121,854'},
 {'지표': '괴리율', '값': '-0.04%'},
 {'지표': '총보수', '값': '0.15%'},
 {'지표': '추적오차율', '값': '0.40%'},
 {'지표': '증권거래', '값': '-'},
 {'지표': '매매차익', '값': '비과세'},
 {'지표': '배당소득세', '값': '-'}]

In [ ]:
import pandas as pd

# 1. set up a list to store data in a dictionary of {indicator: value}
all_rows = []

for code, data_list in results_agg.items():
    # convert to a dictionary of {indicator: value}
    row_dict = {item['지표']: item['값'] for item in data_list}
    row_dict['symbol'] = code

    all_rows.append(row_dict)

# 2. convert to a dataframe
df = pd.DataFrame(all_rows)

# 3. move symbol column to the first column
if 'symbol' in df.columns:
    cols = ['symbol'] + [c for c in df.columns if c != 'symbol']
    df = df[cols]

# 4. redefine columns
df.columns = ['symbol'
            ,'bm_index' # benchmark index   
            ,'lst_date' # listing date
            ,'am_company'  # asset management company
            ,'mkt_capital' # market capitalization
            ,'aum'  # Leveraged Assets Under Management 
            ,'leverage' # leverage
            ,'nav' # Net Asset Value
            ,'dsc_rate' # Discrepancy Rate
            ,'tot_expense' # Total Expense Ratio
            ,'trk_error' # Tracking Error
            , 'stt' # Securities Transaction Tax (STT)
            , 'cgt' # Capital gains tax
            , 'dvt' # Dividend Tax
            ]

# 결과 확인
print(df)


  symbol 기초지수\n(추적지수)            상장일        운용사        시가총액        운용자산 레버리지  \
0    001      코스피 200  2002. 10. 14.  삼성자산운용(주)  26조 4,133억  25조 8,698억   1배   
1    002      코스피 200  2002. 10. 14.  삼성자산운용(주)  26조 4,133억  25조 8,698억   1배   

       NAV     괴리율    총보수  추적오차율 증권거래 매매차익 배당소득세  
0  121,854  -0.04%  0.15%  0.40%    -  비과세     -  
1  121,854  -0.04%  0.15%  0.40%    -  비과세     -  


In [48]:
df.columns

Index(['symbol', '기초지수\n(추적지수)', '상장일', '운용사', '시가총액', '운용자산', '레버리지', 'NAV',
       '괴리율', '총보수', '추적오차율', '증권거래', '매매차익', '배당소득세'],
      dtype='str')

In [50]:
df

,symbol,bm_index,lst_date,am_company,mkt_capital,aum,leverage,nav,dsc_rate,tot_expense,trk_error,stt,cgt,dvt
0,001,코스피 200,2002. 10. 14.,삼성자산운용(주),"26조 4,133억","25조 8,698억",1배,"121,854",-0.04%,0.15%,0.40%,-,비과세,-
1,002,코스피 200,2002. 10. 14.,삼성자산운용(주),"26조 4,133억","25조 8,698억",1배,"121,854",-0.04%,0.15%,0.40%,-,비과세,-


In [4]:
print(table_results)

[{'종목명': '삼성전자', '주식수': '7,022 주', '비중중': '32.90 %', '시세': '290,500', '전일대비': '상승 5,000 (+1.75%)'}, {'종목명': 'SK하이닉스', '주식수': '834 주', '비중중': '25.73 %', '시세': '1,957,000', '전일대비': '상승 77,000 (+4.10%)'}, {'종목명': 'SK스퀘어', '주식수': '139 주', '비중중': '2.70 %', '시세': '1,223,000', '전일대비': '상승 36,000 (+3.03%)'}, {'종목명': '현대차', '주식수': '205 주', '비중중': '2.17 %', '시세': '671,500', '전일대비': '상승 25,500 (+3.95%)'}, {'종목명': '두산에너빌리티', '주식수': '653 주', '비중중': '1.37 %', '시세': '132,900', '전일대비': '상승 4,900 (+3.83%)'}, {'종목명': 'KB금융', '주식수': '482 주', '비중중': '1.25 %', '시세': '158,800', '전일대비': '0 (0.00%)'}, {'종목명': '삼성전기', '주식수': '84 주', '비중중': '1.24 %', '시세': '935,000', '전일대비': '상승 35,000 (+3.89%)'}, {'종목명': '삼성물산', '주식수': '145 주', '비중중': '1.07 %', '시세': '461,000', '전일대비': '상승 9,000 (+1.99%)'}, {'종목명': '한화에어로스페이스', '주식수': '49 주', '비중중': '1.05 %', '시세': '1,305,000', '전일대비': '하락 10,000 (-0.76%)'}, {'종목명': '기아', '주식수': '368 주', '비중중': '1.05 %', '시세': '174,400', '전일대비': '하락 300 (-0.17%)'}]


In [26]:
table_results_agg = dict()
table_results_agg['001'] = table_results
table_results_agg['002'] = table_results
table_results_agg['001']


[{'종목명': '삼성전자',
  '주식수': '7,022 주',
  '비중중': '32.90 %',
  '시세': '290,500',
  '전일대비': '상승 5,000 (+1.75%)'},
 {'종목명': 'SK하이닉스',
  '주식수': '834 주',
  '비중중': '25.73 %',
  '시세': '1,957,000',
  '전일대비': '상승 77,000 (+4.10%)'},
 {'종목명': 'SK스퀘어',
  '주식수': '139 주',
  '비중중': '2.70 %',
  '시세': '1,223,000',
  '전일대비': '상승 36,000 (+3.03%)'},
 {'종목명': '현대차',
  '주식수': '205 주',
  '비중중': '2.17 %',
  '시세': '671,500',
  '전일대비': '상승 25,500 (+3.95%)'},
 {'종목명': '두산에너빌리티',
  '주식수': '653 주',
  '비중중': '1.37 %',
  '시세': '132,900',
  '전일대비': '상승 4,900 (+3.83%)'},
 {'종목명': 'KB금융',
  '주식수': '482 주',
  '비중중': '1.25 %',
  '시세': '158,800',
  '전일대비': '0 (0.00%)'},
 {'종목명': '삼성전기',
  '주식수': '84 주',
  '비중중': '1.24 %',
  '시세': '935,000',
  '전일대비': '상승 35,000 (+3.89%)'},
 {'종목명': '삼성물산',
  '주식수': '145 주',
  '비중중': '1.07 %',
  '시세': '461,000',
  '전일대비': '상승 9,000 (+1.99%)'},
 {'종목명': '한화에어로스페이스',
  '주식수': '49 주',
  '비중중': '1.05 %',
  '시세': '1,305,000',
  '전일대비': '하락 10,000 (-0.76%)'},
 {'종목명': '기아',
  '주식수': '368 주',
  '비중중'

In [ ]:
tbl = pd.DataFrame()
for code, table_results in table_results_agg.items():
    tb = pd.DataFrame(table_results)
    tb['symbol'] = code
    tbl = pd.concat([tbl, tb])

tbl.columns = ['stock_nm', 'stock_qty', 'stock_weight', 'stock_price', 'stock_change']
if 'symbol' in tbl.columns:
    cols = ['symbol'] + [c for c in tbl.columns if c != 'symbol']
    tbl = tbl[cols]
    

In [43]:
pd.DataFrame(tbl)

,종목명,주식수,비중중,시세,전일대비,ID
0,삼성전자,"7,022 주",32.90 %,"290,500","상승 5,000 (+1.75%)",001
1,SK하이닉스,834 주,25.73 %,"1,957,000","상승 77,000 (+4.10%)",001
2,SK스퀘어,139 주,2.70 %,"1,223,000","상승 36,000 (+3.03%)",001
3,현대차,205 주,2.17 %,"671,500","상승 25,500 (+3.95%)",001
4,두산에너빌리티,653 주,1.37 %,"132,900","상승 4,900 (+3.83%)",001
5,KB금융,482 주,1.25 %,"158,800",0 (0.00%),001
6,삼성전기,84 주,1.24 %,"935,000","상승 35,000 (+3.89%)",001
7,삼성물산,145 주,1.07 %,"461,000","상승 9,000 (+1.99%)",001
8,한화에어로스페이스,49 주,1.05 %,"1,305,000","하락 10,000 (-0.76%)",001
9,기아,368 주,1.05 %,"174,400",하락 300 (-0.17%),001


In [27]:
results

[{'지표': '기초지수\n(추적지수)', '값': '코스피 200'},
 {'지표': '상장일', '값': '2002. 10. 14.'},
 {'지표': '운용사', '값': '삼성자산운용(주)'},
 {'지표': '시가총액', '값': '26조 1,322억'},
 {'지표': '운용자산', '값': '24조 8,758억'},
 {'지표': '레버리지', '값': '1배'},
 {'지표': 'NAV', '값': '115,782'},
 {'지표': '괴리율', '값': '0.02%'},
 {'지표': '총보수', '값': '0.15%'},
 {'지표': '추적오차율', '값': '0.41%'},
 {'지표': '증권거래', '값': '-'},
 {'지표': '매매차익', '값': '비과세'},
 {'지표': '배당소득세', '값': '-'}]

In [48]:
from cellect_etf import masterETF
symbols = masterETF()

In [50]:
symbols.df_etf

,Symbol,Category,Name,Price,RiseFall,Change,ChangeRate,NAV,EarningRate,Volume,Amount,MarCap
0,069500,1,KODEX 200,121805,2,5995,5.18,121861.0,54.6965,24857214,3028015,261698
1,360750,4,TIGER 미국S&P500,26980,2,160,0.60,26963.0,8.6117,11207314,302087,173724
2,396500,2,TIGER 반도체TOP10,49400,2,3025,6.52,49423.0,65.4579,34682715,1700620,127995
3,102110,1,TIGER 200,121740,2,5900,5.09,121886.0,54.8572,6566205,800282,104940
4,133690,4,TIGER 미국나스닥100,190635,2,3695,1.98,190314.0,17.4139,958771,182569,99550
...,...,...,...,...,...,...,...,...,...,...,...,...
1094,465620,4,ACE 미국빅테크TOP7 Plus인버스(합성),6320,5,-65,-1.02,6349.0,-14.8099,1607,10,16
1095,253240,3,KIWOOM 200선물인버스,1414,5,-74,-4.97,1415.0,-40.5989,185844,261,14
1096,422260,2,VITA MZ소비액티브,10865,5,-295,-2.64,10957.0,4.3966,7077,78,11
1097,306520,3,HANARO 200선물인버스,2540,5,-125,-4.69,2545.0,-41.2349,29530,75,9


In [3]:
text = '하락 12,500 (-4.38%)'
parts= text.split()
print(parts[2])
number = float(parts[2].replace("%", "").replace(")", "").replace("(", "").replace(",", "").strip())  # 1.35
print(number)

(-4.38%)
-4.38


In [4]:
import FinanceDataReader as fdr

In [5]:
df_etf = fdr.StockListing('ETF/KR')

In [19]:
df_etf

,Symbol,Category,Name,Price,RiseFall,Change,ChangeRate,NAV,EarningRate,Volume,Amount,MarCap
0,069500,1,KODEX 200,124255,2,1600,1.30,124070.0,50.5002,13135219,1628320,263731
1,360750,4,TIGER 미국S&P500,27585,2,175,0.64,27544.0,12.4320,12413131,342247,178144
2,396500,2,TIGER 반도체TOP10,49650,5,-260,-0.52,49633.0,70.4785,16511840,820133,135172
3,102110,1,TIGER 200,124280,2,1590,1.30,124116.0,50.5963,3940288,488846,106694
4,133690,4,TIGER 미국나스닥100,194740,2,1870,0.97,194082.0,22.4596,623953,121627,102862
...,...,...,...,...,...,...,...,...,...,...,...,...
1102,465620,4,ACE 미국빅테크TOP7 Plus인버스(합성),6215,5,-55,-0.88,6215.0,-18.0393,625,3,16
1103,253240,3,KIWOOM 200선물인버스,1392,5,-15,-1.07,1382.0,-39.2225,16122,22,14
1104,422260,2,VITA MZ소비액티브,11495,2,830,7.78,11509.0,-5.5359,8062,92,11
1105,306520,3,HANARO 200선물인버스,2505,5,-50,-1.96,2495.0,-38.2851,6282,15,9


In [ ]:
df_etf.columns

Index(['Symbol', 'Category', 'Name', 'Price', 'RiseFall', 'Change',
       'ChangeRate', 'NAV', 'EarningRate', 'Volume', 'Amount', 'MarCap'],
      dtype='str')